# create-splat — production training

**Project:** `{{PROJECT_NAME}}`  
**Trainer:** nerfstudio splatfacto (Apache-2.0). Commercial use OK.

## What to do

1. `Runtime → Change runtime type → GPU` (T4 is fine — it's the free default).
2. `Runtime → Run all`.
3. Close the tab if you want. The result lands back in your Google Drive.

Expected wall time on a T4: **25–40 min** for 80–200 photos at 30k iterations.

If anything fails, the cell prints the tail of the failing step's log automatically. Paste that into cowork and the next iteration of the skill will target the exact failure.

In [ ]:
# Single-cell production run — install + SfM + train + export + write back to Drive.
#
# Why one cell, no kernel restart: we install nerfstudio into a venv created with
# --system-site-packages, which inherits Colab's preinstalled torch+CUDA but lets
# us pin numpy<2 / protobuf<4 locally without breaking the running kernel. All
# nerfstudio CLI invocations go through the venv's ns-* binaries via subprocess,
# so the host kernel never imports nerfstudio at all — no in-memory ABI conflict,
# no kill-and-restart required.
#
# Headless-Linux fixes baked in (discovered the hard way across four debug rounds):
#   • QT_QPA_PLATFORM=offscreen — Ubuntu's colmap links Qt5 GUI libs.
#   • --no-gpu on ns-process-data — colmap's GPU SIFT needs OpenGL/EGL, Colab has
#     CUDA but no GL display. CPU SIFT works fine, only slightly slower.
#   • --viewer.quit-on-train-completion True — current tyro on Colab requires an
#     explicit True/False value (a bare flag is parsed as missing-value and the
#     *next* argv token, e.g. --vis, gets consumed as the value, causing a parse
#     error: "invalid choice '--vis' … expected one of ('True', 'False')").
#
# Output: writes splat.ply, transforms.json, and a .done marker to
#   /content/drive/MyDrive/splats/{{PROJECT_NAME}}/
# The local create-splat skill watches for .done and continues from there.

import os, sys, subprocess, time, shutil, glob, json
from google.colab import drive

PROJECT = '{{PROJECT_NAME}}'
ITERATIONS = 30000  # bump to 60000 for higher quality (will need ~2× wall time)

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = f'/content/drive/MyDrive/splats/{PROJECT}'
SRC = f'{DRIVE_ROOT}/photos'
OUT = DRIVE_ROOT
assert os.path.isdir(SRC), f'✗ no photos at {SRC} — cowork should have written them here'
n_photos = len([f for f in os.listdir(SRC) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
assert n_photos >= 20, f'✗ only {n_photos} photos at {SRC}'
print(f'project={PROJECT!r}  photos={n_photos}  iters={ITERATIONS}')

# Headless Qt for any subprocess we spawn.
os.environ['QT_QPA_PLATFORM'] = 'offscreen'

def step(label, args, log_path):
    print(f'\n=== {label} ===')
    t0 = time.time()
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1, env=os.environ)
    with open(log_path, 'w') as log:
        for line in p.stdout:
            print(line, end='')
            log.write(line)
    p.wait()
    dt = time.time() - t0
    print(f'\nrc={p.returncode}  ({dt:.0f}s)')
    if p.returncode != 0:
        print(f'\n—— last 60 lines of {log_path} ——')
        subprocess.run(['tail', '-60', log_path])
        raise RuntimeError(f'✗ {label} failed (rc={p.returncode})')

# 1. COLMAP system binary (fast).
step('install colmap',
     ['bash', '-c', 'apt-get update -qq && apt-get install -qq -y colmap'],
     '/content/colmap_install.log')

# 2. Create a venv that inherits system torch+CUDA but isolates nerfstudio deps.
VENV = '/content/venv'
step('create venv',
     [sys.executable, '-m', 'venv', VENV, '--system-site-packages'],
     '/content/venv.log')

# 3. Install nerfstudio + pinned deps inside the venv. The venv's pip will pull
# numpy<2 / protobuf<4 in front of the system versions for any process started
# from the venv. The running kernel keeps its own numpy and never touches
# nerfstudio — so no restart is needed.
step('upgrade venv pip',
     [f'{VENV}/bin/pip', 'install', '--quiet', '--upgrade', 'pip'],
     '/content/pip_upgrade.log')
step('pin numpy<2 protobuf<4 in venv',
     [f'{VENV}/bin/pip', 'install', '--quiet', 'numpy<2', 'protobuf<4'],
     '/content/pin.log')
step('install nerfstudio in venv (slow: 3–7 min)',
     [f'{VENV}/bin/pip', 'install', '--quiet', 'nerfstudio'],
     '/content/nerfstudio_install.log')

# 4. SfM with CPU SIFT (avoids OpenGL requirement of GPU SIFT on headless Colab).
shutil.rmtree('/content/processed', ignore_errors=True)
step('SfM (ns-process-data, ~3–8 min)',
     [f'{VENV}/bin/ns-process-data', 'images',
      '--data', SRC,
      '--output-dir', '/content/processed',
      '--matching-method', 'exhaustive',
      '--no-gpu'],
     '/content/sfm.log')

# Sanity check: transforms.json should have most frames registered.
tj = json.load(open('/content/processed/transforms.json'))
n_registered = len(tj.get('frames', []))
print(f'\nSfM registered {n_registered}/{n_photos} frames')
assert n_registered >= 10, '✗ SfM registered too few frames — capture coverage likely poor'

# 5. Train splatfacto.
shutil.rmtree('/content/runs', ignore_errors=True)
step(f'train splatfacto ({ITERATIONS} iters, ~25–40 min on T4)',
     [f'{VENV}/bin/ns-train', 'splatfacto',
      '--data', '/content/processed',
      '--output-dir', '/content/runs',
      '--max-num-iterations', str(ITERATIONS),
      '--viewer.quit-on-train-completion', 'True',
      '--vis', 'tensorboard'],
     '/content/train.log')

# 6. Find the run config + verify checkpoints exist.
configs = sorted(glob.glob('/content/runs/**/config.yml', recursive=True), key=os.path.getmtime)
assert configs, '✗ no config.yml — training did not start'
CONFIG = configs[-1]
ckpt_dir = os.path.join(os.path.dirname(CONFIG), 'nerfstudio_models')
ckpts = glob.glob(f'{ckpt_dir}/*')
assert ckpts, f'✗ no checkpoints in {ckpt_dir} — training crashed before first save'
print(f'\n✓ training produced {len(ckpts)} checkpoint(s); using config {CONFIG}')

# 7. Export to .ply.
os.makedirs('/content/export', exist_ok=True)
step('export gaussian-splat',
     [f'{VENV}/bin/ns-export', 'gaussian-splat',
      '--load-config', CONFIG,
      '--output-dir', '/content/export'],
     '/content/export.log')
plys = sorted(glob.glob('/content/export/**/*.ply', recursive=True), key=os.path.getmtime)
assert plys, '✗ no .ply in /content/export'
PLY = plys[-1]
print(f'exported {PLY}  ({os.path.getsize(PLY)/1e6:.1f} MB)')

# 8. Write outputs back to Drive — splat.ply, transforms.json, and a .done marker.
# The local create-splat skill polls the .done file every 60s and picks up from there.
os.makedirs(OUT, exist_ok=True)
shutil.copy(PLY, f'{OUT}/splat.ply')
shutil.copy('/content/processed/transforms.json', f'{OUT}/transforms.json')
with open(f'{OUT}/.done', 'w') as f:
    f.write(json.dumps({
        'project': PROJECT,
        'iterations': ITERATIONS,
        'n_photos': n_photos,
        'n_registered': n_registered,
        'ply_size_mb': round(os.path.getsize(f'{OUT}/splat.ply') / 1e6, 1),
        'ckpts': len(ckpts),
    }))
print(f'\n✓ ALL DONE — splat.ply + transforms.json + .done written to {OUT}')
print('  cowork is polling for .done; you can close this tab.')